In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install transformers sentencepiece textblob tqdm

import nltk
nltk.download('stopwords')


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
import os
import glob
import pandas as pd

base_path = "/content/drive/MyDrive/majorproject/tallipdataset"

train_pattern = os.path.join(base_path, "Train", "train_Hindi_Data_*.txt")
test_pattern  = os.path.join(base_path, "Test",  "test_Hindi_Data_*.txt")

train_files = sorted(glob.glob(train_pattern))
test_files  = sorted(glob.glob(test_pattern))

print("Train files:")
for f in train_files:
    print("  ", os.path.basename(f))

print("\nTest files:")
for f in test_files:
    print("  ", os.path.basename(f))


Train files:
   train_Hindi_Data_Business.txt
   train_Hindi_Data_Celebrity_Full.txt
   train_Hindi_Data_Complete_FakeNews.txt
   train_Hindi_Data_Education.txt
   train_Hindi_Data_Entertainment.txt
   train_Hindi_Data_Politics.txt
   train_Hindi_Data_Sports.txt
   train_Hindi_Data_Technology.txt

Test files:
   test_Hindi_Data_Business.txt
   test_Hindi_Data_Celebrity_Full.txt
   test_Hindi_Data_Complete_FakeNews.txt
   test_Hindi_Data_Education.txt
   test_Hindi_Data_Entertainment.txt
   test_Hindi_Data_Politics.txt
   test_Hindi_Data_Sports.txt
   test_Hindi_Data_Technology.txt


In [ ]:
def load_hindi_split(file_list):
    dfs = []
    for fp in file_list:
        # Most likely tab-separated
        df = pd.read_csv(fp, sep='\t', encoding='utf-8')
        dfs.append(df)
    full_df = pd.concat(dfs, ignore_index=True)
    return full_df

train_df = load_hindi_split(train_files)
test_df  = load_hindi_split(test_files)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
train_df.head()


Train shape: (1188, 4)
Test shape: (772, 4)


,Domain,Topic,News,Label
0,Business,कैलिफ़ोर्निया यूफोल्ड्स ऑटो उत्सर्जन मानक,"ट्रम्प के साथ फेस-ऑफ की स्थापना ""कैलिफोर्निया ...",Legit
1,Business,ब्रूडॉग ने नाम में 'पंक' के साथ बार के लिए योज...,"ब्रूडॉग ने अपने नाम में ""पंक"" शब्द का उपयोग कर...",Legit
2,Business,यूरोपीय संघ Applauds Deutsche Boerse के लंदन स...,"""यूरोपीय संघ के नियामकों ने लंदन स्टॉक एक्सचें...",Fake
3,Business,नए टैक्सी कानूनों पर डेनमार्क ऑपरेशन को बंद कर...,Uber ने अगले महीने डेनमार्क में अपना संचालन बं...,Legit
4,Business,अमेरिकन एयरलाइंस ने चीन दक्षिणी के साथ साझेदार...,अमेरिकन एयरलाइंस और यात्रियों द्वारा चीन के सब...,Legit


In [ ]:
# Combine Topic + News into one string
train_df['text'] = train_df['Topic'].astype(str) + " । " + train_df['News'].astype(str)
test_df['text']  = test_df['Topic'].astype(str)  + " । " + test_df['News'].astype(str)

# Labels (assume numeric already; if they are strings you can map)
y_train = train_df['Label'].values
y_test  = test_df['Label'].values

train_texts = train_df['text'].tolist()
test_texts  = test_df['text'].tolist()

print("Train samples:", len(train_texts), " Test samples:", len(test_texts))
print("Example train text:\n", train_texts[0][:300])
print("Example labels:", y_train[:10])


Train samples: 1188  Test samples: 772
Example train text:
 कैलिफ़ोर्निया यूफोल्ड्स ऑटो उत्सर्जन मानक । ट्रम्प के साथ फेस-ऑफ की स्थापना "कैलिफोर्निया की स्वच्छ-वायु एजेंसी ने शुक्रवार को ग्रह-वार्मिंग गैसों को कम करने के लिए राज्य की योजना पर ट्रम्प प्रशासन के साथ संभावित कानूनी लड़ाई की स्थापना के लिए कारों और ट्रकों के लिए सख्त उत्सर्जन मानकों के साथ आगे ब
Example labels: ['Legit' 'Legit' 'Fake' 'Legit' 'Legit' 'Fake' 'Legit' 'Fake' 'Legit'
 'Fake']


In [ ]:
label_map = {'fake':1, 'Fake':1, 'FAKE':1, 'LEGIT':0, 'Legit':0, 'legit':0, 'Legit':0}
y_train = train_df['Label'].map(label_map).values
y_test  = test_df['Label'].map(label_map).values


In [ ]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel
from huggingface_hub import login

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

# Log in to Hugging Face to access gated models
login() # This will prompt you to enter your Hugging Face token

MODEL_NAME = "ai4bharat/indic-bert"   # or "ai4bharat/indic-bert-v1" if this errors

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
indicbert_model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
indicbert_model.eval()

def get_indicbert_embeddings(text_list, batch_size=16, max_length=128):
    all_embeds = []

    for i in range(0, len(text_list), batch_size):
        batch_texts = [str(t) if t is not None else "" for t in text_list[i:i+batch_size]]

        enc = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors='pt'
        ).to(DEVICE)

        with torch.no_grad():
            outputs = indicbert_model(**enc)
            cls_embeddings = outputs.last_hidden_state[:, 0, :]  # [CLS] embedding

        all_embeds.append(cls_embeddings.cpu().numpy())

    return np.vstack(all_embeds)

Using device: cuda


In [ ]:
import numpy as np
from textblob import TextBlob
from nltk.corpus import stopwords
from tqdm import tqdm
import re

class HandcraftedFeatureExtractor:
    def __init__(self, lang='en'):
        self.lang = lang

        if lang == 'hi':
            try:
                self.stop_words = set(stopwords.words('hindi'))
            except:
                self.stop_words = set()

            self.clickbait_words = {
                'शॉकिंग', 'चौंकाने', 'चौंकाने वाला', 'चमत्कार', 'राज़', 'गुप्त',
                'खुलासा', 'अविश्वसनीय', 'सबसे अच्छा', 'सबसे खराब', 'कमाल', 'धमाका',
                'shocking', 'miracle', 'secret', 'confidential',
                'revealed', 'unbelievable', 'best', 'worst', 'amazing'
            }
        else:
            self.stop_words = set(stopwords.words('english'))
            self.clickbait_words = {
                'shocking', 'miracle', 'secret', 'confidential',
                'revealed', 'unbelievable', 'best', 'worst', 'amazing'
            }

    def _tokenize_hi(self, text):
        t = text.strip()
        words = t.split()
        sentences = [s for s in re.split(r'[।.!?]+', t) if s.strip()]
        return words, sentences

    def get_features(self, text_list):
        features = []
        print(f"Extracting 18 stylistic features for {len(text_list)} samples (lang={self.lang})...")

        for text in tqdm(text_list, desc="Handcrafted Extraction"):
            t = str(text).strip()

            if self.lang == 'hi':
                words, sentences = self._tokenize_hi(t)
                blob = None
            else:
                try:
                    blob = TextBlob(t)
                    words = blob.words
                    sentences = blob.sentences
                except:
                    blob = TextBlob("")
                    words, sentences = [], []

            num_words = len(words) if len(words) > 0 else 1
            num_chars = len(t) if len(t) > 0 else 1
            num_sents = len(sentences) if len(sentences) > 0 else 1

            # Structural
            f1 = len(words)
            f2 = len(t)
            f3 = sum(len(w) for w in words) / num_words
            f4 = len(sentences)
            f5 = num_words / num_sents

            # Punctuation / Style
            f6 = t.count('!') / num_chars
            f7 = t.count('?') / num_chars
            f8 = sum(1 for c in t if c.isupper()) / num_chars
            f9 = t.count('"') + t.count("'")

            word_strs = [str(w).lower() for w in words]

            # Lexical
            f10 = sum(1 for w in word_strs if w in self.stop_words) / num_words
            f11 = len(set(word_strs)) / num_words
            f12 = sum(c.isdigit() for c in t) / num_chars

            # Pronouns
            if self.lang == 'hi':
                first_person = {'मैं', 'हम', 'मेरा', 'मेरी', 'मेरे', 'हमारा', 'हमारी', 'हमारे'}
                second_person = {'तुम', 'आप', 'आपका', 'आपकी', 'आपके', 'तेरा', 'तेरी', 'तेरे'}
            else:
                first_person = {'i', 'we', 'my', 'our', 'me', 'us'}
                second_person = {'you', 'your'}

            f13 = len([w for w in word_strs if w in first_person]) / num_words
            f14 = len([w for w in word_strs if w in second_person]) / num_words
            f15 = len([w for w in word_strs if w in self.clickbait_words]) / num_words

            # Sentiment & POS
            if self.lang == 'hi':
                f16, f17, f18 = 0.0, 0.0, 0.0
            else:
                try:
                    f16, f17 = blob.sentiment.polarity, blob.sentiment.subjectivity
                except:
                    f16, f17 = 0.0, 0.0
                try:
                    f18 = len([word for word, tag in blob.tags if tag.startswith('JJ')]) / num_words
                except:
                    f18 = 0.0

            features.append([
                f1, f2, f3, f4, f5,
                f6, f7, f8, f9,
                f10, f11, f12,
                f13, f14, f15,
                f16, f17, f18
            ])

        return np.array(features)


In [ ]:
print("Getting IndicBERT embeddings for TRAIN...")
X_indic_train = get_indicbert_embeddings(train_texts, batch_size=16, max_length=128)
print("Train IndicBERT shape:", X_indic_train.shape)

print("Getting IndicBERT embeddings for TEST...")
X_indic_test = get_indicbert_embeddings(test_texts, batch_size=16, max_length=128)
print("Test IndicBERT shape:", X_indic_test.shape)


Getting IndicBERT embeddings for TRAIN...
Train IndicBERT shape: (1188, 768)
Getting IndicBERT embeddings for TEST...
Test IndicBERT shape: (772, 768)


In [ ]:
extractor = HandcraftedFeatureExtractor(lang='hi')

print("Handcrafted features for TRAIN...")
X_hand_train = extractor.get_features(train_texts)
print("Train handcrafted shape:", X_hand_train.shape)

print("Handcrafted features for TEST...")
X_hand_test = extractor.get_features(test_texts)
print("Test handcrafted shape:", X_hand_test.shape)


Handcrafted features for TRAIN...
Extracting 18 stylistic features for 1188 samples (lang=hi)...


Handcrafted Extraction: 100%|██████████| 1188/1188 [00:01<00:00, 1030.54it/s]


Train handcrafted shape: (1188, 18)
Handcrafted features for TEST...
Extracting 18 stylistic features for 772 samples (lang=hi)...


Handcrafted Extraction: 100%|██████████| 772/772 [00:00<00:00, 1495.90it/s]

Test handcrafted shape: (772, 18)


In [ ]:
# Combine along feature dimension
X_train_all = np.concatenate([X_indic_train, X_hand_train], axis=1)
X_test_all  = np.concatenate([X_indic_test,  X_hand_test],  axis=1)

print("Final train feature shape:", X_train_all.shape)
print("Final test  feature shape:", X_test_all.shape)


Final train feature shape: (1188, 786)
Final test  feature shape: (772, 786)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

clf = LogisticRegression(max_iter=1000, n_jobs=-1)
clf.fit(X_train_all, y_train)

y_pred = clf.predict(X_test_all)

print("✅ Test Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


✅ Test Accuracy: 0.6010362694300518
              precision    recall  f1-score   support

           0       0.58      0.63      0.60       371
           1       0.63      0.57      0.60       401

    accuracy                           0.60       772
   macro avg       0.60      0.60      0.60       772
weighted avg       0.60      0.60      0.60       772



In [ ]:
print(train_df['Label'].unique())
print(test_df['Label'].unique())


['Legit' 'Fake']
['Fake' 'Legit']


In [ ]:
label_map = {
    'FAKE': 1, 'Fake': 1, 'fake': 1,
    'REAL': 0, 'Real': 0, 'real': 0,
    'Legit': 0, 'LEGIT': 0, 'legit': 0
}

train_df['Label_num'] = train_df['Label'].map(label_map)
test_df['Label_num']  = test_df['Label'].map(label_map)

y_train = train_df['Label_num'].values
y_test  = test_df['Label_num'].values


In [ ]:
print(train_df['Domain'].value_counts())
print(test_df['Domain'].value_counts())


Domain
Celebrity        605
Business         105
Politics         100
Education         99
Entertainment     97
Sports            93
Technology        89
Name: count, dtype: int64
Domain
Celebrity        395
Technology        71
Sports            67
Entertainment     63
Education         61
Politics          60
Business          55
Name: count, dtype: int64


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

clf_indic = LogisticRegression(max_iter=1000, n_jobs=-1)
clf_indic.fit(X_indic_train, y_train)

y_pred_indic = clf_indic.predict(X_indic_test)

print("IndicBERT-only Accuracy:", accuracy_score(y_test, y_pred_indic))
print(classification_report(y_test, y_pred_indic))


IndicBERT-only Accuracy: 0.6256476683937824
              precision    recall  f1-score   support

           0       0.60      0.67      0.63       371
           1       0.66      0.58      0.62       401

    accuracy                           0.63       772
   macro avg       0.63      0.63      0.63       772
weighted avg       0.63      0.63      0.63       772



In [ ]:

#  FULL PIPELINE IN ONE CELL


# 1) Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2) Install dependencies
!pip install -q transformers sentencepiece textblob tqdm

import os
import glob
import re
import numpy as np
import pandas as pd
import torch

from transformers import AutoTokenizer, AutoModel
from textblob import TextBlob
from tqdm import tqdm
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report


# 3) Load Hindi TALLIP TXT

base_path = "/content/drive/MyDrive/majorproject/tallipdataset"

train_pattern = os.path.join(base_path, "Train", "train_Hindi_Data_*.txt")
test_pattern  = os.path.join(base_path, "Test",  "test_Hindi_Data_*.txt")

train_files = sorted(glob.glob(train_pattern))
test_files  = sorted(glob.glob(test_pattern))

print("Train files:")
for f in train_files:
    print("  ", os.path.basename(f))

print("\nTest files:")
for f in test_files:
    print("  ", os.path.basename(f))

def load_hindi_split(file_list):
    dfs = []
    for fp in file_list:
        # Most TALLIP files are tab-separated
        df = pd.read_csv(fp, sep='\t', encoding='utf-8')
        dfs.append(df)
    full_df = pd.concat(dfs, ignore_index=True)
    return full_df

train_df = load_hindi_split(train_files)
test_df  = load_hindi_split(test_files)

print("\nTrain shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Columns:", train_df.columns.tolist())


# 4) Build text + labels

# Combine Topic + News as input text
train_df['text'] = train_df['Topic'].astype(str) + " । " + train_df['News'].astype(str)
test_df['text']  = test_df['Topic'].astype(str)  + " । " + test_df['News'].astype(str)

# Map labels to 0/1 if they are strings
print("\nUnique train labels:", train_df['Label'].unique())

if train_df['Label'].dtype == object:
    label_map = {
        'fake': 1, 'Fake': 1, 'FAKE': 1,
        'real': 0, 'Real': 0, 'REAL': 0,
        'legit': 0, 'Legit': 0, 'LEGIT': 0
    }
    train_df['Label_num'] = train_df['Label'].map(label_map)
    test_df['Label_num']  = test_df['Label'].map(label_map)
    if train_df['Label_num'].isna().any() or test_df['Label_num'].isna().any():
        raise ValueError("Some labels did not map to 0/1. Check label_map.")
    y_train = train_df['Label_num'].values
    y_test  = test_df['Label_num'].values
else:
    y_train = train_df['Label'].astype(int).values
    y_test  = test_df['Label'].astype(int).values

train_texts = train_df['text'].tolist()
test_texts  = test_df['text'].tolist()

print("\nTrain samples:", len(train_texts), " Test samples:", len(test_texts))
print("Example text:\n", train_texts[0][:300])
print("Example labels:", y_train[:10])


# 5) Load IndicBERT

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("\nUsing device:", DEVICE)

MODEL_NAME = "ai4bharat/indic-bert"   # if this errors, try "ai4bharat/indic-bert-v1"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
indicbert_model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
indicbert_model.eval()

def get_indicbert_embeddings(text_list, batch_size=8, max_length=256):
    """
    IndicBERT sentence embeddings using MEAN POOLING (better than CLS).
    """
    all_embeds = []

    for i in range(0, len(text_list), batch_size):
        batch_texts = [str(t) if t is not None else "" for t in text_list[i:i+batch_size]]

        enc = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors='pt'
        ).to(DEVICE)

        with torch.no_grad():
            outputs = indicbert_model(**enc)
            last_hidden = outputs.last_hidden_state  # (batch, seq_len, hidden)

            # ---- MEAN POOLING with attention mask ----
            mask = enc['attention_mask'].unsqueeze(-1).expand(last_hidden.size()).float()
            summed = torch.sum(last_hidden * mask, dim=1)
            counts = torch.clamp(mask.sum(dim=1), min=1e-9)
            mean_pool = summed / counts  # (batch, hidden_size)

        all_embeds.append(mean_pool.cpu().numpy())

    return np.vstack(all_embeds)


# 6) Hindi handcrafted features

class HandcraftedFeatureExtractor:
    def __init__(self, lang='en'):
        self.lang = lang

        if lang == 'hi':
            try:
                self.stop_words = set(stopwords.words('hindi'))
            except:
                self.stop_words = set()

            self.clickbait_words = {
                'शॉकिंग', 'चौंकाने', 'चौंकाने वाला', 'चमत्कार', 'राज़', 'गुप्त',
                'खुलासा', 'अविश्वसनीय', 'सबसे अच्छा', 'सबसे खराब', 'कमाल', 'धमाका',
                'shocking', 'miracle', 'secret', 'confidential',
                'revealed', 'unbelievable', 'best', 'worst', 'amazing'
            }
        else:
            self.stop_words = set(stopwords.words('english'))
            self.clickbait_words = {
                'shocking', 'miracle', 'secret', 'confidential',
                'revealed', 'unbelievable', 'best', 'worst', 'amazing'
            }

    def _tokenize_hi(self, text):
        t = text.strip()
        words = t.split()
        sentences = [s for s in re.split(r'[।.!?]+', t) if s.strip()]
        return words, sentences

    def get_features(self, text_list):
        features = []
        print(f"\nExtracting 18 stylistic features for {len(text_list)} samples (lang={self.lang})...")

        for text in tqdm(text_list, desc="Handcrafted Extraction"):
            t = str(text).strip()

            if self.lang == 'hi':
                words, sentences = self._tokenize_hi(t)
                blob = None
            else:
                try:
                    blob = TextBlob(t)
                    words = blob.words
                    sentences = blob.sentences
                except:
                    blob = TextBlob("")
                    words, sentences = [], []

            num_words = len(words) if len(words) > 0 else 1
            num_chars = len(t) if len(t) > 0 else 1
            num_sents = len(sentences) if len(sentences) > 0 else 1

            # Structural
            f1 = len(words)
            f2 = len(t)
            f3 = sum(len(w) for w in words) / num_words
            f4 = len(sentences)
            f5 = num_words / num_sents

            # Punctuation / Style
            f6 = t.count('!') / num_chars
            f7 = t.count('?') / num_chars
            f8 = sum(1 for c in t if c.isupper()) / num_chars
            f9 = t.count('"') + t.count("'")

            word_strs = [str(w).lower() for w in words]

            # Lexical
            f10 = sum(1 for w in word_strs if w in self.stop_words) / num_words
            f11 = len(set(word_strs)) / num_words
            f12 = sum(c.isdigit() for c in t) / num_chars

            # Pronouns
            if self.lang == 'hi':
                first_person = {'मैं', 'हम', 'मेरा', 'मेरी', 'मेरे', 'हमारा', 'हमारी', 'हमारे'}
                second_person = {'तुम', 'आप', 'आपका', 'आपकी', 'आपके', 'तेरा', 'तेरी', 'तेरे'}
            else:
                first_person = {'i', 'we', 'my', 'our', 'me', 'us'}
                second_person = {'you', 'your'}

            f13 = len([w for w in word_strs if w in first_person]) / num_words
            f14 = len([w for w in word_strs if w in second_person]) / num_words
            f15 = len([w for w in word_strs if w in self.clickbait_words]) / num_words

            # Sentiment & POS
            if self.lang == 'hi':
                f16, f17, f18 = 0.0, 0.0, 0.0
            else:
                try:
                    f16, f17 = blob.sentiment.polarity, blob.sentiment.subjectivity
                except:
                    f16, f17 = 0.0, 0.0
                try:
                    f18 = len([word for word, tag in blob.tags if tag.startswith('JJ')]) / num_words
                except:
                    f18 = 0.0

            features.append([
                f1, f2, f3, f4, f5,
                f6, f7, f8, f9,
                f10, f11, f12,
                f13, f14, f15,
                f16, f17, f18
            ])

        return np.array(features)


# 7) Compute embeddings

print("\nGetting IndicBERT embeddings (mean pooling, max_length=256) for TRAIN...")
X_indic_train = get_indicbert_embeddings(train_texts, batch_size=8, max_length=256)
print("Train IndicBERT shape:", X_indic_train.shape)

print("\nGetting IndicBERT embeddings (mean pooling, max_length=256) for TEST...")
X_indic_test = get_indicbert_embeddings(test_texts, batch_size=8, max_length=256)
print("Test IndicBERT shape:", X_indic_test.shape)

# -------------------------
# 8) Handcrafted features
# -------------------------
extractor = HandcraftedFeatureExtractor(lang='hi')

print("\nHandcrafted features for TRAIN...")
X_hand_train = extractor.get_features(train_texts)
print("Train handcrafted shape:", X_hand_train.shape)

print("\nHandcrafted features for TEST...")
X_hand_test = extractor.get_features(test_texts)
print("Test handcrafted shape:", X_hand_test.shape)

# -------------------------
# 9) Train & evaluate models
# -------------------------

# (A) IndicBERT-only + LinearSVC
scaler_indic = StandardScaler()
X_train_indic_scaled = scaler_indic.fit_transform(X_indic_train)
X_test_indic_scaled  = scaler_indic.transform(X_indic_test)

svm_indic = LinearSVC()
svm_indic.fit(X_train_indic_scaled, y_train)
y_pred_indic = svm_indic.predict(X_test_indic_scaled)

print("\n==============================")
print("IndicBERT-only (mean pooled) + SVM")
print("==============================")
print("Accuracy:", accuracy_score(y_test, y_pred_indic))
print(classification_report(y_test, y_pred_indic))

# (B) IndicBERT + Handcrafted combined
X_train_all = np.concatenate([X_indic_train, X_hand_train], axis=1)
X_test_all  = np.concatenate([X_indic_test,  X_hand_test],  axis=1)

scaler_all = StandardScaler()
X_train_all_scaled = scaler_all.fit_transform(X_train_all)
X_test_all_scaled  = scaler_all.transform(X_test_all)

svm_all = LinearSVC()
svm_all.fit(X_train_all_scaled, y_train)
y_pred_all = svm_all.predict(X_test_all_scaled)

print("\n==============================")
print("IndicBERT + Handcrafted + SVM")
print("==============================")
print("Accuracy:", accuracy_score(y_test, y_pred_all))
print(classification_report(y_test, y_pred_all))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Train files:
   train_Hindi_Data_Business.txt
   train_Hindi_Data_Celebrity_Full.txt
   train_Hindi_Data_Complete_FakeNews.txt
   train_Hindi_Data_Education.txt
   train_Hindi_Data_Entertainment.txt
   train_Hindi_Data_Politics.txt
   train_Hindi_Data_Sports.txt
   train_Hindi_Data_Technology.txt

Test files:
   test_Hindi_Data_Business.txt
   test_Hindi_Data_Celebrity_Full.txt
   test_Hindi_Data_Complete_FakeNews.txt
   test_Hindi_Data_Education.txt
   test_Hindi_Data_Entertainment.txt
   test_Hindi_Data_Politics.txt
   test_Hindi_Data_Sports.txt
   test_Hindi_Data_Technology.txt

Train shape: (1188, 4)
Test shape: (772, 4)
Columns: ['Domain', 'Topic', 'News', 'Label']

Unique train labels: ['Legit' 'Fake']

Train samples: 1188  Test samples: 772
Example text:
 कैलिफ़ोर्निया यूफोल्ड्स ऑटो उत्सर्जन मानक । ट्रम्प के साथ फेस-ऑफ की स्थापना "कैलिफोर्निया की स्वच्छ-वायु एजेंसी ने शुक्रवार को ग्रह-वार्मिंग गैसों को कम करने के लिए राज्य की योजना पर ट्रम्प प्रशासन के साथ संभावित कानूनी लड़ाई क

Handcrafted Extraction: 100%|██████████| 1188/1188 [00:00<00:00, 4298.17it/s]


Train handcrafted shape: (1188, 18)

Handcrafted features for TEST...

Extracting 18 stylistic features for 772 samples (lang=hi)...


Handcrafted Extraction: 100%|██████████| 772/772 [00:00<00:00, 4545.96it/s]


Test handcrafted shape: (772, 18)

IndicBERT-only (mean pooled) + SVM
Accuracy: 0.8341968911917098
              precision    recall  f1-score   support

           0       0.83      0.82      0.83       371
           1       0.83      0.85      0.84       401

    accuracy                           0.83       772
   macro avg       0.83      0.83      0.83       772
weighted avg       0.83      0.83      0.83       772


IndicBERT + Handcrafted + SVM
Accuracy: 0.844559585492228
              precision    recall  f1-score   support

           0       0.86      0.81      0.83       371
           1       0.83      0.88      0.85       401

    accuracy                           0.84       772
   macro avg       0.85      0.84      0.84       772
weighted avg       0.85      0.84      0.84       772



In [ ]:

#   DOMAIN-WISE ACCURACY, PRECISION, RECALL, F1


from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# y_test → true labels for test set
# y_pred_all → predicted labels from your final model
# test_df['Domain'] → domain column for each test sample

domains = sorted(test_df['Domain'].unique())

domain_results = []

print("\n=========== DOMAIN-WISE PERFORMANCE ===========\n")
for d in domains:
    idx = test_df.index[test_df['Domain'] == d].tolist()

    true_d = y_test[idx]
    pred_d = y_pred_all[idx]

    acc = accuracy_score(true_d, pred_d)
    prec = precision_score(true_d, pred_d, zero_division=0)
    rec  = recall_score(true_d, pred_d, zero_division=0)
    f1   = f1_score(true_d, pred_d, zero_division=0)

    domain_results.append([d, len(idx), acc, prec, rec, f1])

    print(f"### {d.upper()} ###")
    print(f"Samples   : {len(idx)}")
    print(f"Accuracy  : {acc*100:.2f}%")
    print(f"Precision : {prec*100:.2f}%")
    print(f"Recall    : {rec*100:.2f}%")
    print(f"F1-score  : {f1*100:.2f}%")
    print("-------------------------------------------")


# OPTIONAL: turn into a DataFrame for display
import pandas as pd



=========== DOMAIN-WISE PERFORMANCE ===========

### BUSINESS ###
Samples   : 55
Accuracy  : 74.55%
Precision : 76.92%
Recall    : 71.43%
F1-score  : 74.07%
-------------------------------------------
### CELEBRITY ###
Samples   : 395
Accuracy  : 82.78%
Precision : 80.53%
Recall    : 88.35%
F1-score  : 84.26%
-------------------------------------------
### EDUCATION ###
Samples   : 61
Accuracy  : 90.16%
Precision : 87.88%
Recall    : 93.55%
F1-score  : 90.62%
-------------------------------------------
### ENTERTAINMENT ###
Samples   : 63
Accuracy  : 90.48%
Precision : 93.33%
Recall    : 87.50%
F1-score  : 90.32%
-------------------------------------------
### POLITICS ###
Samples   : 60
Accuracy  : 90.00%
Precision : 93.33%
Recall    : 87.50%
F1-score  : 90.32%
-------------------------------------------
### SPORTS ###
Samples   : 67
Accuracy  : 85.07%
Precision : 84.62%
Recall    : 89.19%
F1-score  : 86.84%
-------------------------------------------
### TECHNOLOGY ###
Samples   : 7

In [ ]:

#   HINDI FAKE NEWS CLASSIFICATION — INDICBERTv2-MLM-only
#   (No v1. Only pure v2. Cleanest version.)

# 1) Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# 2) Install dependencies
!pip install -q transformers sentencepiece textblob tqdm datasets

import os, glob, re, random
import numpy as np
import pandas as pd
import torch

from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
from textblob import TextBlob
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from datasets import Dataset

# ------------------------- Seed -------------------------
def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
set_seed(42)


# 3) Load Hindi Train + Test (TALLIP TXT domain files)


base = "/content/drive/MyDrive/majorproject/tallipdataset"
train_files = sorted(glob.glob(base + "/Train/train_Hindi_Data_*.txt"))
test_files  = sorted(glob.glob(base + "/Test/test_Hindi_Data_*.txt"))

def load_txt(files):
    dfs = []
    for f in files:
        df = pd.read_csv(f, sep="\t", encoding="utf-8")
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True)

train_df = load_txt(train_files)
test_df  = load_txt(test_files)

print("Train:", train_df.shape, "Test:", test_df.shape)
print("Columns:", train_df.columns.tolist())


# 4) Cleaning + Domain Token + Label Mapping


def clean(text):
    if not isinstance(text, str): text = str(text)
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[\"\'\t\r]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

train_df["Domain_token"] = "[" + train_df["Domain"].str.upper() + "] "
test_df["Domain_token"]  = "[" + test_df["Domain"].str.upper() + "] "

train_df["raw_text"] = train_df["Topic"].astype(str) + " । " + train_df["News"].astype(str)
test_df["raw_text"]  = test_df["Topic"].astype(str) + " । " + test_df["News"].astype(str)

train_df["text"] = (train_df["Domain_token"] + train_df["raw_text"]).apply(clean)
test_df["text"]  = (test_df["Domain_token"]  + test_df["raw_text"]).apply(clean)

# ------ Label mapping ------
label_map = {
    "fake":1, "Fake":1, "FAKE":1,
    "real":0, "Real":0, "REAL":0,
    "legit":0, "Legit":0, "LEGIT":0
}

train_df["Label_num"] = train_df["Label"].map(label_map).astype(int)
test_df["Label_num"]  = test_df["Label"].map(label_map).astype(int)

y_train = train_df["Label_num"].values
y_test  = test_df["Label_num"].values

train_texts = train_df["text"].tolist()
test_texts  = test_df["text"].tolist()

print("Example cleaned text:\n", train_texts[0][:250])


# 5) Load IndicBERT v2 (MLM-only)


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

MODEL_NAME = "ai4bharat/IndicBERTv2-MLM-only"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
indicbert = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
indicbert.eval()

def encode(text_list, max_length=256, batch=8):
    all_emb = []
    for i in range(0, len(text_list), batch):
        chunk = text_list[i:i+batch]
        enc = tokenizer(chunk, padding=True, truncation=True,
                        max_length=max_length, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            outputs = indicbert(**enc)
            H = outputs.last_hidden_state  # (B, L, 768)

            mask = enc["attention_mask"].unsqueeze(-1).float()
            summed = (H * mask).sum(1)
            counts = mask.sum(1)
            emb = (summed / counts).cpu().numpy()
        all_emb.append(emb)
    return np.vstack(all_emb)

print("\nEncoding TRAIN with IndicBERTv2 ...")
X_indic_train = encode(train_texts)
print("Encoding TEST with IndicBERTv2 ...")
X_indic_test  = encode(test_texts)

print("Embeddings shape:", X_indic_train.shape)

# 6) Hindi Handcrafted Features


class HFExtractor:
    def __init__(self):
        try:
            self.stop = set(stopwords.words("hindi"))
        except:
            self.stop = set()
        self.clickbait = {
            "चौंकाने","शॉकिंग","चमत्कार","गुप्त","राज़","खुलासा",
            "अविश्वसनीय","सबसे","कमाल","धमाका",
            "shocking","secret","miracle","amazing","unbelievable"
        }

    def tokenize_hi(self, t):
        w = t.split()
        s = [x for x in re.split(r"[।.!?]+", t) if x.strip()]
        return w, s

    def get(self, texts):
        feats = []
        for t in tqdm(texts):
            t = str(t).strip()
            words, sents = self.tokenize_hi(t)

            nw = len(words) if len(words)>0 else 1
            nc = len(t) if len(t)>0 else 1
            ns = len(sents) if len(sents)>0 else 1

            wlow = [w.lower() for w in words]

            f = [
                len(words),
                len(t),
                sum(len(w) for w in words)/nw,
                len(sents),
                nw/ns,
                t.count("!")/nc,
                t.count("?")/nc,
                sum(c.isupper() for c in t)/nc,
                t.count('"') + t.count("'"),
                sum(w in self.stop for w in wlow)/nw,
                len(set(wlow))/nw,
                sum(c.isdigit() for c in t)/nc,
                sum(w in {"मैं","हम","मेरा","मेरी","हमारा"} for w in wlow)/nw,
                sum(w in {"आप","तुम","आपका","तेरा"} for w in wlow)/nw,
                sum(w in self.clickbait for w in wlow)/nw,
                0.0, 0.0, 0.0
            ]
            feats.append(f)
        return np.array(feats)

print("\nExtracting handcrafted features ...")
hf = HFExtractor()
X_hand_train = hf.get(train_texts)
X_hand_test  = hf.get(test_texts)

# ==========================================================
# 7) Combine + Train SVM
# ==========================================================

X_train = np.concatenate([X_indic_train, X_hand_train], axis=1)
X_test  = np.concatenate([X_indic_test,  X_hand_test],  axis=1)

sc = StandardScaler()
X_train_s = sc.fit_transform(X_train)
X_test_s  = sc.transform(X_test)

clf = LinearSVC()
clf.fit(X_train_s, y_train)

y_pred = clf.predict(X_test_s)

print("\n=========== OVERALL RESULTS ===========")
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

# ==========================================================
# 8) DOMAIN-WISE PERFORMANCE
# ==========================================================

domains = sorted(test_df["Domain"].unique())
results = []

print("\n=========== DOMAIN-WISE PERFORMANCE ===========\n")
for d in domains:
    idx = test_df.index[test_df["Domain"] == d].tolist()
    true = y_test[idx]
    pred = y_pred[idx]

    acc = accuracy_score(true, pred)
    prec = precision_score(true, pred, zero_division=0)
    rec  = recall_score(true, pred, zero_division=0)
    f1   = f1_score(true, pred, zero_division=0)

    print(f"### {d.upper()} ###")
    print(f"Samples    : {len(idx)}")
    print(f"Accuracy   : {acc*100:.2f}%")
    print(f"Precision  : {prec*100:.2f}%")
    print(f"Recall     : {rec*100:.2f}%")
    print(f"F1-score   : {f1*100:.2f}%")
    print("---------------------------------------")

    results.append([d, len(idx), acc, prec, rec, f1])

df_domain = pd.DataFrame(results,
    columns=["Domain","Samples","Accuracy","Precision","Recall","F1-score"])
df_domain


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Train: (1188, 4) Test: (772, 4)
Columns: ['Domain', 'Topic', 'News', 'Label']
Example cleaned text:
 [BUSINESS] कैलिफ़ोर्निया यूफोल्ड्स ऑटो उत्सर्जन मानक । ट्रम्प के साथ फेस-ऑफ की स्थापना कैलिफोर्निया की स्वच्छ-वायु एजेंसी ने शुक्रवार को ग्रह-वार्मिंग गैसों को कम करने के लिए राज्य की योजना पर ट्रम्प प्रशासन के साथ संभावित कानूनी लड़ाई की स्थापना के 
Device: cuda


tokenizer_config.json:   0%|          | 0.00/51.0 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.75M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]


Encoding TRAIN with IndicBERTv2 ...
Encoding TEST with IndicBERTv2 ...
Embeddings shape: (1188, 768)

Extracting handcrafted features ...


100%|██████████| 772/772 [00:00<00:00, 3887.62it/s]
/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(



=========== OVERALL RESULTS ===========
Accuracy: 0.8601036269430051
              precision    recall  f1-score   support

           0       0.87      0.84      0.85       371
           1       0.85      0.88      0.87       401

    accuracy                           0.86       772
   macro avg       0.86      0.86      0.86       772
weighted avg       0.86      0.86      0.86       772


=========== DOMAIN-WISE PERFORMANCE ===========

### BUSINESS ###
Samples    : 55
Accuracy   : 78.18%
Precision  : 78.57%
Recall     : 78.57%
F1-score   : 78.57%
---------------------------------------
### CELEBRITY ###
Samples    : 395
Accuracy   : 84.81%
Precision  : 84.76%
Recall     : 86.41%
F1-score   : 85.58%
---------------------------------------
### EDUCATION ###
Samples    : 61
Accuracy   : 96.72%
Precision  : 93.94%
Recall     : 100.00%
F1-score   : 96.88%
---------------------------------------
### ENTERTAINMENT ###
Samples    : 63
Accuracy   : 90.48%
Precision  : 88.24%
Recall     :

,Domain,Samples,Accuracy,Precision,Recall,F1-score
0,Business,55,0.781818,0.785714,0.785714,0.785714
1,Celebrity,395,0.848101,0.847619,0.864078,0.855769
2,Education,61,0.967213,0.939394,1.000000,0.968750
3,Entertainment,63,0.904762,0.882353,0.937500,0.909091
4,Politics,60,0.933333,0.937500,0.937500,0.937500
5,Sports,67,0.791045,0.794872,0.837838,0.815789
6,Technology,71,0.859155,0.837838,0.885714,0.861111


In [ ]:

# SAVE MODEL, FEATURES, REPORTS, AND PREDICTIONS (ONE BLOCK)


import os
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report

# 1. Create save folder in Drive
save_dir = "/content/drive/MyDrive/majorproject/saved_model"
os.makedirs(save_dir, exist_ok=True)
print("Saving to:", save_dir)

# 2. Save SVM classifier + scaler
joblib.dump(clf, f"{save_dir}/svm_model.pkl")
joblib.dump(sc, f"{save_dir}/scaler.pkl")
print("SVM model + scaler saved!")

# 3. Save IndicBERTv2 embeddings
np.save(f"{save_dir}/X_indic_train.npy", X_indic_train)
np.save(f"{save_dir}/X_indic_test.npy", X_indic_test)
print("IndicBERT embeddings saved!")

# 4. Save handcrafted features
np.save(f"{save_dir}/X_hand_train.npy", X_hand_train)
np.save(f"{save_dir}/X_hand_test.npy", X_hand_test)
print("Handcrafted features saved!")

# 5. Save domain-wise results
df_domain.to_csv(f"{save_dir}/domain_wise_results.csv", index=False)
print("Domain-wise results saved!")

# 6. Save overall classification report
report_text = classification_report(y_test, y_pred)
with open(f"{save_dir}/classification_report.txt", "w", encoding="utf-8") as f:
    f.write(report_text)
print("Classification report saved!")

# 7. Save predictions
results_df = pd.DataFrame({
    "Text": test_texts,
    "True_Label": y_test,
    "Predicted_Label": y_pred,
    "Domain": test_df["Domain"]
})
results_df.to_csv(f"{save_dir}/predictions.csv", index=False)
print("Predictions saved!")

print("\n✅ ALL FILES SAVED SUCCESSFULLY!")


Saving to: /content/drive/MyDrive/majorproject/saved_model
SVM model + scaler saved!
IndicBERT embeddings saved!
Handcrafted features saved!
Domain-wise results saved!
Classification report saved!
Predictions saved!

✅ ALL FILES SAVED SUCCESSFULLY!


In [ ]:

# LOADER BLOCK: LOAD MODEL, FEATURES, AND EVALUATE AGAIN


import os
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# 1. Set save directory (same as used while saving)
save_dir = "/content/drive/MyDrive/majorproject/saved_model"
print("Loading from:", save_dir)

# 2. Load trained SVM model and scaler
clf = joblib.load(f"{save_dir}/svm_model.pkl")
sc  = joblib.load(f"{save_dir}/scaler.pkl")
print("✅ Loaded SVM model and scaler")

# 3. Load saved features (IndicBERT embeddings + handcrafted)
X_indic_test = np.load(f"{save_dir}/X_indic_test.npy")
X_hand_test  = np.load(f"{save_dir}/X_hand_test.npy")
print("✅ Loaded X_indic_test and X_hand_test")

# 4. Rebuild test feature matrix and scale
X_test = np.concatenate([X_indic_test, X_hand_test], axis=1)
X_test_s = sc.transform(X_test)
print("✅ Combined and scaled test features:", X_test_s.shape)

# 5. Load predictions metadata (texts, true labels, domains)
pred_df = pd.read_csv(f"{save_dir}/predictions.csv")
test_texts = pred_df["Text"].tolist()
y_test     = pred_df["True_Label"].values
domains    = pred_df["Domain"].values
print("✅ Loaded predictions.csv with", len(pred_df), "rows")

# 6. Recompute predictions from loaded model (sanity check)
y_pred = clf.predict(X_test_s)

print("\n=========== OVERALL RESULTS (RELOADED MODEL) ===========")
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

# 7. Domain-wise evaluation using reloaded predictions
unique_domains = sorted(pred_df["Domain"].unique())
domain_results = []

print("\n=========== DOMAIN-WISE PERFORMANCE (RELOADED) ===========\n")
for d in unique_domains:
    idx = pred_df.index[pred_df["Domain"] == d].tolist()
    true_d = y_test[idx]
    pred_d = y_pred[idx]

    acc = accuracy_score(true_d, pred_d)
    prec = precision_score(true_d, pred_d, zero_division=0)
    rec  = recall_score(true_d, pred_d, zero_division=0)
    f1   = f1_score(true_d, pred_d, zero_division=0)

    domain_results.append([d, len(idx), acc, prec, rec, f1])

    print(f"### {d.upper()} ###")
    print(f"Samples    : {len(idx)}")
    print(f"Accuracy   : {acc*100:.2f}%")
    print(f"Precision  : {prec*100:.2f}%")
    print(f"Recall     : {rec*100:.2f}%")
    print(f"F1-score   : {f1*100:.2f}%")
    print("---------------------------------------")

df_domain_loaded = pd.DataFrame(
    domain_results,
    columns=["Domain", "Samples", "Accuracy", "Precision", "Recall", "F1-score"]
)

print("\n=========== DOMAIN-WISE TABLE (RELOADED) ===========")
print(df_domain_loaded)


Loading from: /content/drive/MyDrive/majorproject/saved_model
✅ Loaded SVM model and scaler
✅ Loaded X_indic_test and X_hand_test
✅ Combined and scaled test features: (772, 786)
✅ Loaded predictions.csv with 772 rows

=========== OVERALL RESULTS (RELOADED MODEL) ===========
Accuracy: 0.8601036269430051
              precision    recall  f1-score   support

           0       0.87      0.84      0.85       371
           1       0.85      0.88      0.87       401

    accuracy                           0.86       772
   macro avg       0.86      0.86      0.86       772
weighted avg       0.86      0.86      0.86       772


=========== DOMAIN-WISE PERFORMANCE (RELOADED) ===========

### BUSINESS ###
Samples    : 55
Accuracy   : 78.18%
Precision  : 78.57%
Recall     : 78.57%
F1-score   : 78.57%
---------------------------------------
### CELEBRITY ###
Samples    : 395
Accuracy   : 84.81%
Precision  : 84.76%
Recall     : 86.41%
F1-score   : 85.58%
---------------------------------------


In [ ]:

# Save loader.py inside the SAME saved_model folder


loader_code = r'''
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# Location of saved model folder
save_dir = "/content/drive/MyDrive/majorproject/saved_model"

print("Loading model & data from:", save_dir)

# 1. Load model and scaler
clf = joblib.load(f"{save_dir}/svm_model.pkl")
sc  = joblib.load(f"{save_dir}/scaler.pkl")
print("✅ Loaded SVM model and scaler")

# 2. Load features
X_indic_test = np.load(f"{save_dir}/X_indic_test.npy")
X_hand_test  = np.load(f"{save_dir}/X_hand_test.npy")
X_test = np.concatenate([X_indic_test, X_hand_test], axis=1)
X_test_scaled = sc.transform(X_test)
print("✅ Loaded & scaled feature matrices")

# 3. Load stored test metadata
pred_df = pd.read_csv(f"{save_dir}/predictions.csv")
test_texts = pred_df["Text"].tolist()
y_test = pred_df["True_Label"].values
domains = pred_df["Domain"].values

print(f"✅ Loaded predictions.csv with {len(pred_df)} rows")

# 4. Predict again
y_pred = clf.predict(X_test_scaled)

print("\\n=========== OVERALL RESULTS (RELOADED MODEL) ===========")
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

# 5. Domain-wise:
unique_domains = sorted(pred_df['Domain'].unique())
print("\\n=========== DOMAIN-WISE PERFORMANCE ===========\\n")

for dom in unique_domains:
    idx = pred_df.index[pred_df["Domain"] == dom]
    y_t = y_test[idx]
    y_p = y_pred[idx]

    acc = accuracy_score(y_t, y_p)
    prec = precision_score(y_t, y_p, zero_division=0)
    rec = recall_score(y_t, y_p, zero_division=0)
    f1 = f1_score(y_t, y_p, zero_division=0)

    print(f"### {dom.upper()} ###")
    print(f"Samples    : {len(idx)}")
    print(f"Accuracy   : {acc*100:.2f}%")
    print(f"Precision  : {prec*100:.2f}%")
    print(f"Recall     : {rec*100:.2f}%")
    print(f"F1-score   : {f1*100:.2f}%")
    print("---------------------------------------")
'''

# Write loader.py to Drive
save_dir = "/content/drive/MyDrive/majorproject/saved_model"
file_path = f"{save_dir}/loader.py"

with open(file_path, "w", encoding="utf-8") as f:
    f.write(loader_code)

print("✅ loader.py saved at:", file_path)


✅ loader.py saved at: /content/drive/MyDrive/majorproject/saved_model/loader.py
